In [1]:
import pandas as pd
import numpy as np
import time
import surprise
from surprise import Reader, Dataset, SVD, NMF, KNNBasic
from surprise.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import os
import kagglehub
import json
import gc


class YelpLoader:
    def __init__(self, sample_size=100000):
        self.sample_size = sample_size
        
        # 1. Download/Locate Data
        # kagglehub downloads to a specific cache, or uses /kaggle/input if valid
        print("Locating Yelp Dataset...")
        try:
            self.data_dir = kagglehub.dataset_download("yelp-dataset/yelp-dataset")
        except:
            # Fallback if running directly on Kaggle kernel
            self.data_dir = "/kaggle/input/yelp-dataset"
            
        self.business_path = os.path.join(self.data_dir, 'yelp_academic_dataset_business.json')
        self.review_path = os.path.join(self.data_dir, 'yelp_academic_dataset_review.json')
        
        # 2. Lazy Load Execution
        self.valid_business_ids = set()
        self.movies_df = self._load_businesses_lazy()
        self.ratings_df = self._load_reviews_lazy()
        
        # 3. Memory Optimization: Filter Metadata
        # We only need metadata for items that are actually in the ratings sample
        print("Filtering business metadata to match sample...")
        valid_items = set(self.ratings_df['item_id'].unique())
        self.movies_df = self.movies_df[self.movies_df['item_id'].isin(valid_items)].copy()
        print(f"Filtered businesses: {len(self.movies_df)} items")
        
        # Force garbage collection to free up any temp memory
        gc.collect()

    def _load_businesses_lazy(self):
        """
        Reads business.json line-by-line. 
        Only stores Title + Category in memory.
        """
        print("Lazy loading businesses...")
        data_list = []
        
        with open(self.business_path, 'r', encoding='utf-8') as f:
            for line in f:
                # Load one line only
                record = json.loads(line)
                
                # Filter immediately: Must have categories
                if record.get('categories'):
                    bid = record['business_id']
                    self.valid_business_ids.add(bid) # Add to set for O(1) lookup
                    
                    data_list.append({
                        'item_id': bid,
                        'title': record['name'],
                        'genre_str': record['categories']
                    })
        
        return pd.DataFrame(data_list)

    def _review_generator(self):
        """
        Generator function that yields one cleaned review tuple at a time.
        Zero memory overhead for the file itself.
        """
        with open(self.review_path, 'r', encoding='utf-8') as f:
            for line in f:
                record = json.loads(line)
                
                # Check if this review belongs to a valid business
                if record['business_id'] in self.valid_business_ids:
                    # Yield tuple: (user, item, rating)
                    yield (
                        record['user_id'], 
                        record['business_id'], 
                        float(record['stars'])
                    )

    def _load_reviews_lazy(self):
        """
        Consumes the generator up to sample_size.
        """
        print(f"Lazy loading top {self.sample_size} valid reviews...")
        
        # Create iterator
        review_iter = self._review_generator()
        
        # Collect exactly 'sample_size' items
        data = []
        try:
            for _ in range(self.sample_size):
                data.append(next(review_iter))
        except StopIteration:
            print("Reached end of file before sample_size.")
            
        # Convert to DataFrame with optimized types
        df = pd.DataFrame(data, columns=['user_id', 'item_id', 'rating'])
        
        # Memory Optimization: Downcast float64 to float32
        df['rating'] = df['rating'].astype('float32')
        
        print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        return df

    def get_surprise_train_test(self, test_size=0.2):
        print("Converting to Surprise Dataset...")
        reader = Reader(rating_scale=(1, 5))
        data = Dataset.load_from_df(self.ratings_df, reader)
        return train_test_split(data, test_size=test_size, random_state=42)

    def get_content_data(self):
        return self.movies_df

In [2]:
# --- Base Recommender Class ---
class BaseRecommender:
    def fit(self, trainset):
        raise NotImplementedError
    
    def test(self, testset):
        raise NotImplementedError

# --- Collaborative Filtering Wrapper (Surprise Models) ---
class CFRecommender(BaseRecommender):
    def __init__(self, algorithm_name='svd', sim_options=None):
        self.algo_name = algorithm_name
        if algorithm_name == 'svd':
            self.model = SVD()
        elif algorithm_name == 'nmf':
            self.model = NMF()
        elif algorithm_name == 'user_based':
            self.model = KNNBasic(sim_options={'user_based': True, 'name': 'cosine'})
        elif algorithm_name == 'item_based':
            self.model = KNNBasic(sim_options={'user_based': False, 'name': 'cosine'})
        else:
            raise ValueError("Unknown CF Algorithm")
            
    def fit(self, trainset):
        self.model.fit(trainset)
        
    def test(self, testset):
        # Surprise's test method returns a list of Prediction objects
        return self.model.test(testset)
    
    def predict(self, uid, iid):
        return self.model.predict(uid, iid).est
# --- Content-Based Recommender (Similarity Logic) ---
class ContentBasedRecommender(BaseRecommender):
    def __init__(self, movies_df):
        self.movies_df = movies_df
        self.item_ids = movies_df['item_id'].values
        # Map item_id to row index in the TF-IDF matrix
        self.item_to_index = {iid: idx for idx, iid in enumerate(self.item_ids)}
        self.tfidf_matrix = None
        self.user_profiles = defaultdict(list)
        
    def fit(self, trainset):
        # 1. Build Item Similarity Matrix (Sparse)
        print("Building TF-IDF matrix...")
        tfidf = TfidfVectorizer(stop_words='english', dtype=np.float32)
        # We store the sparse matrix. It is much smaller than the dense similarity matrix.
        self.tfidf_matrix = tfidf.fit_transform(self.movies_df['genre_str'])
        
        # 2. Build User Profiles from Trainset
        # Store Indices of items user liked (rating >= 3.5)
        print("Building user profiles...")
        for uid, iid, rating in trainset.all_ratings():
            # Convert inner IDs to raw IDs
            raw_uid = trainset.to_raw_uid(uid)
            raw_iid = trainset.to_raw_iid(iid)
            
            if rating >= 3.5:
                # Store the MATRIX INDEX, not the ID, for faster lookup
                if raw_iid in self.item_to_index:
                    idx = self.item_to_index[raw_iid]
                    self.user_profiles[raw_uid].append(idx)
                
    def predict_score(self, uid, iid):
        # On-the-fly similarity calculation
        if iid not in self.item_to_index:
            return 0
        
        target_idx = self.item_to_index[iid]
        liked_item_indices = self.user_profiles.get(uid, [])
        
        if not liked_item_indices:
            return 0 # Cold start user
            
        # Get vectors from the sparse matrix
        # target_vec is (1, n_features)
        target_vec = self.tfidf_matrix[target_idx]
        
        # user_vecs is (n_liked, n_features)
        # Slicing a sparse matrix by row indices is efficient
        user_vecs = self.tfidf_matrix[liked_item_indices]
        
        # Compute cosine similarity between Target and ALL Liked items at once
        # Result shape: (1, n_liked)
        similarities = cosine_similarity(target_vec, user_vecs)
        
        if similarities.shape[1] == 0:
            return 0
            
        max_sim = similarities.max()
                    
        # Scale 0-1 similarity to 1-5 rating scale
        predicted_rating = 1 + (max_sim * 4) 
        return predicted_rating

    def test(self, testset):
        predictions = []
        for uid, iid, true_r in testset:
            est = self.predict_score(uid, iid)
            predictions.append(surprise.prediction_algorithms.predictions.Prediction(
                uid, iid, true_r, est, details={'was_impossible': False}
            ))
        return predictions
# # --- Content-Based Recommender (Similarity Logic) ---
# class ContentBasedRecommender(BaseRecommender):
#     def __init__(self, movies_df):
#         self.movies_df = movies_df
#         self.item_ids = movies_df['item_id'].values
#         self.tfidf_matrix = None
#         self.cosine_sim = None
#         self.user_profiles = defaultdict(list)
    #     self.item_to_index = {iid: idx for idx, iid in enumerate(self.item_ids)}
        
    # def fit(self, trainset):
    #     # 1. Build Item Similarity Matrix
    #     print("Building TF-IDF matrix...")
    #     tfidf = TfidfVectorizer(stop_words='english', dtype=np.float32)
    #     self.tfidf_matrix = tfidf.fit_transform(self.movies_df['genre_str'])
        
    #     print("Computing cosine similarity...")
    #     # Compute similarity and immediately cast/free memory
    #     # Note: resulting matrix is dense, ensuring float32 reduces size by 50%
    #     self.cosine_sim = cosine_similarity(self.tfidf_matrix, self.tfidf_matrix).astype('float32')
        
    #     # Free up the large TF-IDF matrix immediately
    #     del self.tfidf_matrix
    #     gc.collect()
        
    #     # 2. Build User Profiles from Trainset
    #     # Store items user liked (rating >= 3.5) to use for similarity lookup
    #     for uid, iid, rating in trainset.all_ratings():
    #         # Convert inner IDs to raw IDs
    #         raw_uid = trainset.to_raw_uid(uid)
    #         raw_iid = trainset.to_raw_iid(iid)
    #         if rating >= 3.5:
    #             self.user_profiles[raw_uid].append(raw_iid)
                
    # def predict_score(self, uid, iid):
    #     # "Use the max similarities to get the items"
    #     if iid not in self.item_to_index:
    #         return 0
        
    #     target_idx = self.item_to_index[iid]
    #     user_liked_items = self.user_profiles.get(uid, [])
        
    #     if not user_liked_items:
    #         return 0 # Cold start user
            
    #     max_sim = 0
    #     for liked_item in user_liked_items:
    #         if liked_item in self.item_to_index:
    #             liked_idx = self.item_to_index[liked_item]
    #             sim = self.cosine_sim[target_idx, liked_idx]
    #             if sim > max_sim:
    #                 max_sim = sim
                    
    #     # Scale 0-1 similarity to 1-5 rating scale for consistency with CF metrics
    #     predicted_rating = 1 + (max_sim * 4) 
    #     return predicted_rating

    # def test(self, testset):
    #     predictions = []
    #     for uid, iid, true_r in testset:
    #         est = self.predict_score(uid, iid)
    #         # Create a Surprise-compatible Prediction object
    #         predictions.append(surprise.prediction_algorithms.predictions.Prediction(
    #             uid, iid, true_r, est, details={'was_impossible': False}
    #         ))
    #     return predictions

# --- Hybrid Recommender (Weighted SVD + Content) ---
# class HybridRecommender(BaseRecommender):
#     def __init__(self, svd_model, content_model, alpha=0.5):
#         self.svd = svd_model
#         self.content = content_model
#         self.alpha = alpha # Weight for SVD
        
#     def fit(self, trainset):
#         # We assume individual models are already fitted or fit them here
#         self.svd.fit(trainset)
#         self.content.fit(trainset)
        
#     def test(self, testset):
#         predictions = []
#         for uid, iid, true_r in testset:
#             final_score = self.predict(uid, iid)
#             predictions.append(surprise.prediction_algorithms.predictions.Prediction(
#                 uid, iid, true_r, final_score, details={'was_impossible': False}
#             ))
#         return predictions

    # def predict(self, uid, iid):
    #     # SVD Score
    #     svd_score = self.svd.predict(uid, iid)
        
    #     # Content Score
    #     content_score = self.content.predict_score(uid, iid)
        
    #     # Weighted Hybrid
    #     return (self.alpha * svd_score) + ((1 - self.alpha) * content_score)




# --- Hybrid Recommender (Weighted CF + Content) ---
class HybridRecommender(BaseRecommender):
    def __init__(self, cf_model, content_model, alpha=0.5):
        """
        Args:
            cf_model: Collaborative filtering model (e.g., CFRecommender)
            content_model: Content-based model
            alpha: Weight for CF model (1-alpha for content model)
        """
        self.cf = cf_model
        self.content = content_model
        self.alpha = alpha
        self.prediction_stats = {'cf_fail': 0, 'cb_fail': 0, 'both_fail': 0, 'success': 0}
        
    def fit(self, trainset):
        """Fit both models"""
        self.cf.fit(trainset)
        self.content.fit(trainset)
    
    def predict(self, uid, iid):
        """
        Make hybrid prediction
        Returns Prediction-like object with .est attribute
        """
        cf_score = None
        cb_score = None
        
        # Get CF prediction
        try:
            cf_pred = self.cf.predict(uid, iid)
            cf_score = cf_pred.est if hasattr(cf_pred, 'est') else cf_pred
        except Exception as e:
            self.prediction_stats['cf_fail'] += 1
        
        # Get Content-Based prediction
        try:
            cb_pred = self.content.predict(uid, iid)
            cb_score = cb_pred.est if hasattr(cb_pred, 'est') else cb_pred
        except Exception as e:
            self.prediction_stats['cb_fail'] += 1
        
        # Combine scores
        if cf_score is not None and cb_score is not None:
            # Both worked - weighted average
            final_score = (self.alpha * cf_score) + ((1 - self.alpha) * cb_score)
            self.prediction_stats['success'] += 1
        elif cf_score is not None:
            # Only CF worked
            final_score = cf_score
            self.prediction_stats['success'] += 1
        elif cb_score is not None:
            # Only CB worked
            final_score = cb_score
            self.prediction_stats['success'] += 1
        else:
            # Both failed - return default
            final_score = 2.5  # Middle of 1-5 scale
            self.prediction_stats['both_fail'] += 1
        
        # Return Prediction-like object
        class PredictionResult:
            def __init__(self, est):
                self.est = est
        
        return PredictionResult(final_score)
    
    def test(self, testset):
        """Test method for compatibility"""
        from surprise.prediction_algorithms.predictions import Prediction
        
        predictions = []
        for uid, iid, true_r in testset:
            pred = self.predict(uid, iid)
            est = pred.est if hasattr(pred, 'est') else pred
            
            predictions.append(Prediction(
                uid, iid, true_r, est, details={'was_impossible': False}
            ))
        return predictions
    
    def print_diagnostics(self):
        """Print prediction statistics"""
        total = sum(self.prediction_stats.values())
        if total > 0:
            print("\n=== Hybrid Model Diagnostics ===")
            print(f"Total predictions: {total}")
            print(f"Successful: {self.prediction_stats['success']} ({100*self.prediction_stats['success']/total:.1f}%)")
            print(f"CF failures: {self.prediction_stats['cf_fail']} ({100*self.prediction_stats['cf_fail']/total:.1f}%)")
            print(f"CB failures: {self.prediction_stats['cb_fail']} ({100*self.prediction_stats['cb_fail']/total:.1f}%)")
            print(f"Both failed: {self.prediction_stats['both_fail']} ({100*self.prediction_stats['both_fail']/total:.1f}%)")


In [3]:
# # # --- 5. Evaluator (Memory Safe & Scale Adapted) ---
# # class Evaluator:
# #     def evaluate_model(self, model, trainset, testset, name):
# #         print(f"Testing {name}...")
# #         start_fit = time.time()
# #         model.fit(trainset)
# #         fit_time = time.time() - start_fit
        
# #         testset.sort(key=lambda x: x[0])
# #         start_test = time.time()
        
# #         rmse_sse, count, sum_precision, sum_recall, n_users = 0, 0, 0, 0, 0
# #         current_uid, user_preds = None, []
        
# #         # THRESHOLD UPDATE: Used 7.0 as 'Good' because scale is 1-10
# #         def process_user_buffer(preds, k=10, threshold=7.0):
# #             if not preds: return 0, 0
# #             preds.sort(key=lambda x: x[0], reverse=True)
# #             n_rel = sum((true_r >= threshold) for (_, true_r) in preds)
# #             n_rec_k = sum((est >= threshold) for (est, _) in preds[:k])
# #             n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) 
# #                                   for (est, true_r) in preds[:k])
# #             prec = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
# #             rec = n_rel_and_rec_k / n_rel if n_rel != 0 else 0
# #             return prec, rec

# #         for uid, iid, true_r in testset:
# #             if hasattr(model, 'predict'):
# #                 p = model.predict(uid, iid)
# #                 est = p.est if hasattr(p, 'est') else p
# #             else:
# #                 est = model.predict_score(uid, iid)
            
# #             rmse_sse += (est - true_r) ** 2
# #             count += 1
            
# #             if uid != current_uid:
# #                 if current_uid is not None:
# #                     p, r = process_user_buffer(user_preds)
# #                     sum_precision += p
# #                     sum_recall += r
# #                     n_users += 1
# #                 current_uid = uid
# #                 user_preds = []
# #             user_preds.append((est, true_r))
            
# #         if current_uid is not None and user_preds:
# #             p, r = process_user_buffer(user_preds)
# #             sum_precision += p
# #             sum_recall += r
# #             n_users += 1
            
# #         inference_time = time.time() - start_test
# #         rmse = np.sqrt(rmse_sse / count) if count > 0 else 0
# #         avg_precision = sum_precision / n_users if n_users > 0 else 0
# #         avg_recall = sum_recall / n_users if n_users > 0 else 0
        
# #         return {
# #             "Algorithm": name,
# #             "RMSE": round(rmse, 4),
# #             "Precision@10": round(avg_precision, 4),
# #             "Recall@10": round(avg_recall, 4),
# #             "Inference Time (s)": round(inference_time, 4)
# #         }



# # --- 5. Evaluator (Corrected Metrics) ---
# class Evaluator:
#     def evaluate_model(self, model, trainset, testset, name):
#         print(f"Testing {name}...")
        
#         # Training phase
#         start_fit = time.time()
#         model.fit(trainset)
#         fit_time = time.time() - start_fit
        
#         # Sort test set by user ID for per-user metrics
#         testset.sort(key=lambda x: x[0])
        
#         # Inference phase - measure ONLY prediction time
#         start_test = time.time()
#         predictions = []
#         for uid, iid, true_r in testset:
#             if hasattr(model, 'predict'):
#                 p = model.predict(uid, iid)
#                 est = p.est if hasattr(p, 'est') else p
#             else:
#                 est = model.predict_score(uid, iid)
#             predictions.append((uid, iid, est, true_r))
#         inference_time = time.time() - start_test
        
#         # Calculate RMSE
#         rmse_sse = sum((est - true_r) ** 2 for (_, _, est, true_r) in predictions)
#         count = len(predictions)
#         rmse = np.sqrt(rmse_sse / count) if count > 0 else 0
        
#         # Calculate Precision@10 and Recall@10 per user
#         sum_precision, sum_recall, n_users = 0, 0, 0
#         current_uid = None
#         user_preds = []
        
#         def process_user_buffer(preds, k=10, threshold=7.0):
#             """
#             Calculate Precision@k and Recall@k for one user.
            
#             Precision@k: fraction of top-k recommendations that are relevant
#             Recall@k: fraction of all relevant items that appear in top-k
#             """
#             if not preds:
#                 return 0, 0
            
#             # Sort by estimated score (descending) to get top-k recommendations
#             preds.sort(key=lambda x: x[0], reverse=True)
            
#             # Get top-k predictions
#             top_k = preds[:k]
            
#             # Count relevant items in top-k (based on TRUE ratings)
#             n_relevant_in_top_k = sum((true_r >= threshold) for (est, true_r) in top_k)
            
#             # Count total relevant items for this user (across all their test items)
#             n_relevant_total = sum((true_r >= threshold) for (est, true_r) in preds)
            
#             # Precision@k = relevant items in top-k / k
#             prec = n_relevant_in_top_k / min(k, len(top_k)) if len(top_k) > 0 else 0
            
#             # Recall@k = relevant items in top-k / total relevant items
#             rec = n_relevant_in_top_k / n_relevant_total if n_relevant_total > 0 else 0
            
#             return prec, rec
        
#         # Process predictions per user
#         for uid, iid, est, true_r in predictions:
#             if uid != current_uid:
#                 # Process previous user's predictions
#                 if current_uid is not None:
#                     p, r = process_user_buffer(user_preds)
#                     sum_precision += p
#                     sum_recall += r
#                     n_users += 1
                
#                 # Start new user
#                 current_uid = uid
#                 user_preds = []
            
#             user_preds.append((est, true_r))
        
#         # Process last user
#         if current_uid is not None and user_preds:
#             p, r = process_user_buffer(user_preds)
#             sum_precision += p
#             sum_recall += r
#             n_users += 1
        
#         # Calculate averages
#         avg_precision = sum_precision / n_users if n_users > 0 else 0
#         avg_recall = sum_recall / n_users if n_users > 0 else 0
        
#         return {
#             "Algorithm": name,
#             "RMSE": round(rmse, 4),
#             "Precision@10": round(avg_precision, 4),
#             "Recall@10": round(avg_recall, 4),
#             "Fit Time (s)": round(fit_time, 4),
#             "Inference Time (s)": round(inference_time, 4)
#         }


# --- 5. Evaluator (TRUE Recommendation Evaluation) ---
class Evaluator:
    def evaluate_model(self, model, trainset, testset, name, k=10, threshold=7.0):
        """
        Evaluates recommendation system properly:
        1. Train on trainset
        2. For each user, recommend K items from ALL items they haven't seen in training
        3. Check if these recommendations appear in testset with high ratings
        
        Args:
            model: Recommendation model
            trainset: Surprise trainset object
            testset: List of (user, item, rating) tuples
            name: Model name
            k: Number of recommendations
            threshold: Rating threshold for "relevant" items
        """
        print(f"Training {name}...")
        
        # Training phase
        start_fit = time.time()
        model.fit(trainset)
        fit_time = time.time() - start_fit
        print(f"  Training completed in {fit_time:.2f}s")
        
        # Get all items and users
        all_items = set(trainset.all_items())
        all_item_ids = [trainset.to_raw_iid(i) for i in all_items]
        
        # Build ground truth from testset
        print(f"  Building ground truth...")
        test_user_items = defaultdict(dict)  # {user: {item: rating}}
        for uid, iid, rating in testset:
            test_user_items[uid][iid] = rating
        
        # Build training user-item map (to exclude already seen items)
        train_user_items = defaultdict(set)
        for uid, iid, rating in trainset.all_ratings():
            raw_uid = trainset.to_raw_uid(uid)
            raw_iid = trainset.to_raw_iid(iid)
            train_user_items[raw_uid].add(raw_iid)
        
        # Get users that appear in test set
        test_users = list(test_user_items.keys())
        
        print(f"  Generating recommendations for {len(test_users)} users...")
        
        # Generate recommendations and evaluate
        start_inference = time.time()
        
        sum_precision, sum_recall, n_users = 0, 0, 0
        sum_ndcg, sum_hit_rate = 0, 0
        
        for user_idx, user in enumerate(test_users):
            if (user_idx + 1) % 100 == 0:
                print(f"    Progress: {user_idx + 1}/{len(test_users)} users")
            
            # Get items user hasn't seen in training
            seen_items = train_user_items.get(user, set())
            candidate_items = [item for item in all_item_ids if item not in seen_items]
            
            # Predict scores for all unseen items
            predictions = []
            for item in candidate_items:
                try:
                    if hasattr(model, 'predict'):
                        pred = model.predict(user, item)
                        est = pred.est if hasattr(pred, 'est') else pred
                    else:
                        est = model.predict_score(user, item)
                    predictions.append((item, est))
                except:
                    # Skip items that can't be predicted
                    continue
            
            # Sort by predicted score and get top-k
            predictions.sort(key=lambda x: x[1], reverse=True)
            top_k_items = [item for item, score in predictions[:k]]
            
            # Get relevant items from test set (items with rating >= threshold)
            relevant_items = {item for item, rating in test_user_items[user].items() 
                            if rating >= threshold}
            
            # Calculate metrics
            if len(relevant_items) > 0 and len(top_k_items) > 0:
                # Hits: relevant items that appear in top-k recommendations
                hits = [item for item in top_k_items if item in relevant_items]
                n_hits = len(hits)
                
                # Precision@k = hits / k
                precision = n_hits / k
                
                # Recall@k = hits / total_relevant
                recall = n_hits / len(relevant_items)
                
                # Hit Rate: did we recommend at least one relevant item?
                hit_rate = 1.0 if n_hits > 0 else 0.0
                
                # NDCG@k
                dcg = sum([1.0 / np.log2(idx + 2) for idx, item in enumerate(top_k_items) 
                          if item in relevant_items])
                idcg = sum([1.0 / np.log2(idx + 2) for idx in range(min(k, len(relevant_items)))])
                ndcg = dcg / idcg if idcg > 0 else 0.0
                
                sum_precision += precision
                sum_recall += recall
                sum_hit_rate += hit_rate
                sum_ndcg += ndcg
                n_users += 1
        
        inference_time = time.time() - start_inference
        
        # Calculate RMSE on testset
        print(f"  Calculating RMSE...")
        rmse_sse = 0
        count = 0
        for uid, iid, true_r in testset:
            try:
                if hasattr(model, 'predict'):
                    pred = model.predict(uid, iid)
                    est = pred.est if hasattr(pred, 'est') else pred
                else:
                    est = model.predict_score(uid, iid)
                rmse_sse += (est - true_r) ** 2
                count += 1
            except:
                continue
        
        rmse = np.sqrt(rmse_sse / count) if count > 0 else 0
        
        # Calculate averages
        avg_precision = sum_precision / n_users if n_users > 0 else 0
        avg_recall = sum_recall / n_users if n_users > 0 else 0
        avg_hit_rate = sum_hit_rate / n_users if n_users > 0 else 0
        avg_ndcg = sum_ndcg / n_users if n_users > 0 else 0
        
        print(f"  Evaluation completed!\n")
        
        return {
            "Algorithm": name,
            "RMSE": round(rmse, 4),
            f"Precision@{k}": round(avg_precision, 4),
            f"Recall@{k}": round(avg_recall, 4),
            f"NDCG@{k}": round(avg_ndcg, 4),
            f"Hit Rate@{k}": round(avg_hit_rate, 4),
            "Fit Time (s)": round(fit_time, 4),
            "Inference Time (s)": round(inference_time, 4)
        }

In [4]:
# --- MAIN EXECUTION ---

# 1. Prepare Data (Auto-downloads from Kaggle)
loader = YelpLoader(sample_size=100000) 

trainset, testset = loader.get_surprise_train_test()
movies_df = loader.get_content_data()


Locating Yelp Dataset...
Lazy loading businesses...
Lazy loading top 100000 valid reviews...
Memory Usage: 15.45 MB
Filtering business metadata to match sample...
Filtered businesses: 9970 items
Converting to Surprise Dataset...


In [5]:
# 2. Define Models
models_to_test = [
    # CFRecommender('user_based'),
    # CFRecommender('item_based'),
    CFRecommender('svd'),
    CFRecommender('nmf'),
    ContentBasedRecommender(movies_df)
]

In [6]:
# 3. Initialize Evaluator
evaluator = Evaluator()
results = []


In [7]:
# # 4. Run Individual Models
# print("before loop")
# for model in models_to_test:
#     print("in loop")
#     name = getattr(model, 'algo_name', 'Content-Based')
#     res = evaluator.evaluate_model(model, trainset, testset, name, k=10, threshold = 3)
#     results.append(res)
# 4. Run Individual Models
print("before loop")

for model in models_to_test:
    print("in loop")
    name = getattr(model, 'algo_name', 'Content-Based')
    
    # Run the evaluation
    res = evaluator.evaluate_model(model, trainset, testset, name, k=10, threshold = 3)
    
    # --- NEW: Print the result immediately ---
    print(f"--- Results for {name} ---")
    print(res)
    print("-" * 30) # Optional separator line for readability
    # -----------------------------------------

    results.append(res)

before loop
in loop
Training svd...
  Training completed in 1.79s
  Building ground truth...
  Generating recommendations for 18562 users...
    Progress: 100/18562 users
    Progress: 200/18562 users
    Progress: 300/18562 users
    Progress: 400/18562 users
    Progress: 500/18562 users
    Progress: 600/18562 users
    Progress: 700/18562 users
    Progress: 800/18562 users
    Progress: 900/18562 users
    Progress: 1000/18562 users
    Progress: 1100/18562 users
    Progress: 1200/18562 users
    Progress: 1300/18562 users
    Progress: 1400/18562 users
    Progress: 1500/18562 users
    Progress: 1600/18562 users
    Progress: 1700/18562 users
    Progress: 1800/18562 users
    Progress: 1900/18562 users
    Progress: 2000/18562 users
    Progress: 2100/18562 users
    Progress: 2200/18562 users
    Progress: 2300/18562 users
    Progress: 2400/18562 users
    Progress: 2500/18562 users
    Progress: 2600/18562 users
    Progress: 2700/18562 users
    Progress: 2800/18562 users


In [8]:
# 5. Run Hybrid Model (Weighted SVD + Content)
print("Building Hybrid Model...")
cf_part = CFRecommender('svd')
cb_part = ContentBasedRecommender(movies_df)

hybrid = HybridRecommender(cf_part, cb_part, alpha=0.6) # 60% SVD, 40% Content
res_hybrid = evaluator.evaluate_model(hybrid, trainset, testset, "Hybrid (SVD+Content)", k=10, threshold = 3)
results.append(res_hybrid)

Building Hybrid Model...
Training Hybrid (SVD+Content)...
Building TF-IDF matrix...
Building user profiles...
  Training completed in 2.18s
  Building ground truth...
  Generating recommendations for 18562 users...
    Progress: 100/18562 users
    Progress: 200/18562 users
    Progress: 300/18562 users
    Progress: 400/18562 users
    Progress: 500/18562 users
    Progress: 600/18562 users
    Progress: 700/18562 users
    Progress: 800/18562 users
    Progress: 900/18562 users
    Progress: 1000/18562 users
    Progress: 1100/18562 users
    Progress: 1200/18562 users
    Progress: 1300/18562 users
    Progress: 1400/18562 users
    Progress: 1500/18562 users
    Progress: 1600/18562 users
    Progress: 1700/18562 users
    Progress: 1800/18562 users
    Progress: 1900/18562 users
    Progress: 2000/18562 users
    Progress: 2100/18562 users
    Progress: 2200/18562 users
    Progress: 2300/18562 users
    Progress: 2400/18562 users
    Progress: 2500/18562 users
    Progress: 2600/

In [9]:
# 6. Final Output
results_df = pd.DataFrame(results)
print("\n--- Final Evaluation Results ---")
print(results_df)

# Save to CSV
results_df.to_csv("recommender_benchmark_results.csv", index=False)
print("Results saved to 'recommender_benchmark_results.csv'")


--- Final Evaluation Results ---
              Algorithm    RMSE  Precision@10  Recall@10  NDCG@10  \
0                   svd  1.2566        0.0006     0.0057   0.0028   
1                   nmf  1.4063        0.0003     0.0029   0.0013   
2         Content-Based  3.7894        0.0004     0.0035   0.0018   
3  Hybrid (SVD+Content)  1.2569        0.0006     0.0061   0.0029   

   Hit Rate@10  Fit Time (s)  Inference Time (s)  
0       0.0061        1.7889            817.3971  
1       0.0030        6.5988            743.4534  
2       0.0037        0.3771          24294.7327  
3       0.0065        2.1839           2580.9038  
Results saved to 'recommender_benchmark_results.csv'


In [10]:
models_to_test